In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
from Utils.rfmap import load_rf_maps, RFMapList, asrfmap, plot_2d_rfmap
from Utils.plotting import apply_light_plot_style
from scipy.ndimage import gaussian_filter
from matplotlib import pyplot as plt

apply_light_plot_style()

In [ ]:
base_dir: Path = Path("/mnt/senzailab/Kai/#Recording/m19/")
date = 260827

sessionID = 5
probeList = ("A",)
targetProbe = "A"
rf_time_range = (0.0, 0.2)
# cluster_forming_z = 1.5

is_save = True

session_dir = base_dir / str(date)
hd_session = session_dir / f"{date}_9"
rf_session = session_dir / f"{date}_{sessionID}"
save_path = rf_session / "data/rfmapping/"

In [ ]:
rf_maps_by_probe = {}
summed_rf_maps_by_probe = {}
rf_centers_by_probe = {}
rfs_by_probe = {}

for probe in probeList:
    rf_source_path = (
            rf_session
            / "data/rfmapping/good/-100_400_1ms"
            / f"Probe{probe}"
            # / f"regular_unitsSpikeCounts_{date}_{sessionID}.rfmap"
            / f"rotation_30_unitsSpikeCounts_{date}_{sessionID}.rfmap"

    )
    raw = load_rf_maps(rf_source_path)
    summed = raw.sum(*rf_time_range)
    # Existing QC: this drops a unit if any pooled spatial bin is zero.
    zero_unit_offsets = np.unique(summed.where(0)[0])
    keep_unit_mask = np.ones(summed.n_units, dtype=bool, )
    keep_unit_mask[zero_unit_offsets] = False

    excluded_unit_ids = np.asarray(summed.unit_ids)[zero_unit_offsets]
    raw = RFMapList([rf_map for rf_map, keep in zip(
        raw, keep_unit_mask, strict=True, ) if keep], raw.source_path, )

    summed = raw.sum(*rf_time_range)

    print(f"Kept {summed.n_units} in Probe{probe}")

    result_path = rf_source_path.with_suffix(".npz")
    result = summed.rf_2d(
        is_shuffle=False,
        result_path=result_path,
    )
    center = summed.rf_2d(
        is_shuffle=False,
        is_center=True,
        result_path=result_path,
    )

    rf_maps_by_probe[probe] = raw
    summed_rf_maps_by_probe[probe] = summed
    rfs_by_probe[probe] = result
    rf_centers_by_probe[probe] = center

rf_all = np.concatenate(
    [rfs_by_probe[probe] for probe in probeList],
    axis=0,
)

center_all = np.concatenate(
    [rf_centers_by_probe[probe] for probe in probeList],
    axis=0,
)

# Keep probe identity because unit IDs may overlap between probes.
RFunitPool = [
    (probe, unit_id)
    for probe in probeList
    for unit_id in rf_maps_by_probe[probe].unit_ids
]

print({probe: rfs_by_probe[probe].shape[0] for probe in probeList})
num_neurons_with_rf = np.count_nonzero(center_all)
print(num_neurons_with_rf)

In [ ]:
"""target_unitID = 372
target_maps = summed_rf_maps_by_probe[targetProbe]
target_unit = target_maps.by_unit_id(target_unitID)
target_unit_index = target_maps.unit_ids.index(target_unitID)

target_unit_bump = rfs_by_probe[targetProbe][
    target_unit_index
]
target_unit_bump_x = np.any(target_unit_bump, axis=0).astype(np.uint8)

plot_2d_rfmap(target_unit.to_2d_array(), cmap="grey")

plot_2d_rfmap(target_unit_bump, cmap="grey")"""

In [ ]:
for index, rf in enumerate(rfs_by_probe[targetProbe]):
    probe, unit = RFunitPool[index]
    print(f"({probe}, {unit})")
    plot_2d_rfmap(rf, cmap="grey", is_save=is_save, save_path=f"{save_path}/rfmap/units/{probe}/{unit}.png")
    plot_2d_rfmap(rf, cmap="grey", is_save=is_save, save_path=f"{save_path}/rfmap/units/{probe}/{unit}.svg")


In [ ]:
rf_counts_by_probe = {}
rf_counts = np.zeros(rf_all.shape[1:], dtype=np.int64)

for probe in probeList:
    probe_counts = rfs_by_probe[probe].sum(axis=0, dtype=np.int64)
    rf_counts_by_probe[probe] = probe_counts
    rf_counts += probe_counts
    print("------------ Probe", probe, "------------")
    plot_2d_rfmap(probe_counts, cmap="grey", is_save=is_save, save_path=f"{save_path}/rfmap/{probe}/rf.png")
    plot_2d_rfmap(probe_counts, cmap="grey", is_save=is_save, save_path=f"{save_path}/rfmap/{probe}/rf.svg")


rf_counts_smoothed = gaussian_filter(
    rf_counts.astype(float),
    sigma=(1.0, 1.0),
    mode="nearest",
)

print("------------ All Probes ------------")
plot_2d_rfmap(rf_counts, cmap="grey", is_save=is_save, save_path=f"{save_path}/rfmap/all/rf.png")
plot_2d_rfmap(rf_counts, cmap="grey", is_save=is_save, save_path=f"{save_path}/rfmap/all/rf.svg")

plot_2d_rfmap(rf_counts_smoothed, cmap="grey", is_save=is_save, save_path=f"{save_path}/rfmap/all/rf_smoothed.png")
plot_2d_rfmap(rf_counts_smoothed, cmap="grey", is_save=is_save, save_path=f"{save_path}/rfmap/all/rf_smoothed.svg")


In [ ]:
rf_counts_by_probe = {}
rf_counts = np.zeros(center_all.shape[1:], dtype=np.int64)

for probe in probeList:
    probe_counts = rf_centers_by_probe[probe].sum(axis=0, dtype=np.int64)
    rf_counts_by_probe[probe] = probe_counts
    rf_counts += probe_counts
    print("------------ Probe", probe, "------------")
    plot_2d_rfmap(probe_counts, cmap="grey")

rf_counts_smoothed = gaussian_filter(
    rf_counts.astype(float),
    sigma=(1.0, 1.0),
    mode="nearest",
)

print("------------ All Probes ------------")
plot_2d_rfmap(rf_counts, cmap="grey", is_save=is_save, save_path=f"{save_path}/rfmap/center/all/rf.png")
plot_2d_rfmap(rf_counts, cmap="grey", is_save=is_save, save_path=f"{save_path}/rfmap/center/all/rf.svg")

plot_2d_rfmap(rf_counts_smoothed, cmap="grey", is_save=is_save, save_path=f"{save_path}/rfmap/center/all/rf_smoothed.png")
plot_2d_rfmap(rf_counts_smoothed, cmap="grey", is_save=is_save, save_path=f"{save_path}/rfmap/center/all/rf_smoothed.svg")

In [ ]:
plt.plot(asrfmap(rf_counts).to_1d_array("x"))
plt.title("RF Maps horizontal")
if is_save:
    plt.savefig(f"{save_path}/rfmap/all/horizontal.png")
plt.show()
plt.plot(asrfmap(rf_counts).to_1d_array("y"))
plt.title("RF Maps vertical")
if is_save:
    plt.savefig(f"{save_path}/rfmap/all/vertical.png")
plt.show()
